# Healthcare Analysis & Disease Prediction Model
## End-to-End Data Analytics Pipeline

**Micro Project | SY AIML AIDS | School of Engineering and Technology**

| Detail | Info |
|--------|------|
| Topic | Healthcare Analysis & Disease Prediction |
| Domain | Healthcare |
| Dataset | 10,500 Patient Records (Synthetic, Clinically Modelled) |
| Target | Disease Prediction (Binary Classification) |
| Tools | Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn |

---
## Step 1: Problem Definition

**Problem Statement**: Healthcare systems globally struggle with early and accurate disease detection. Late diagnosis of diabetes and heart disease leads to poor patient outcomes and high treatment costs.

**Aim**: Develop an end-to-end data analytics pipeline for healthcare data that enables early disease prediction.

**Objectives**:
1. Perform comprehensive EDA on 10,500 patient records
2. Identify significant clinical patterns and correlations
3. Apply hypothesis testing (H0: No significant relationship, H1: Significant relationship exists)
4. Build predictive models: Linear Regression, Logistic Regression, Decision Tree

---
## Step 2: Data Collection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                              roc_auc_score, roc_curve, mean_squared_error, r2_score,
                              ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
print("All libraries loaded successfully!")

In [ ]:
# Load the generated dataset
df = pd.read_csv("../data/healthcare_dataset.csv")
print(f"Dataset Shape: {df.shape}")
print(f"Disease Prevalence: {df["Disease"].mean():.2%}")
print(f"Diabetes Prevalence: {df["Diabetes"].mean():.2%}")
print(f"Heart Disease: {df["Heart_Disease"].mean():.2%}")
df.head()

---
## Step 3: Data Cleaning

In [ ]:
print("Missing Values:")
print(df.isnull().sum()[df.isnull().sum()>0])

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ["float64","int64"]:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

print("After cleaning:", df.isnull().sum().sum(), "missing values")
print("Duplicates:", df.duplicated().sum())

---
## Step 4: Exploratory Data Analysis

In [ ]:
num_cols = ["Age","BMI","Systolic_BP","Diastolic_BP","Heart_Rate",
            "Blood_Glucose","Cholesterol","Hemoglobin","LDL","HDL",
            "Triglycerides","HbA1c","Creatinine"]
df[num_cols].describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle("Distribution of Key Clinical Variables", fontsize=16, fontweight="bold")
cols_plot = ["Age","BMI","Systolic_BP","Diastolic_BP","Blood_Glucose","Cholesterol",
             "HbA1c","LDL","HDL","Triglycerides","Heart_Rate","Hemoglobin"]
for ax, col in zip(axes.flatten(), cols_plot):
    ax.hist(df[col], bins=40, color="#2196F3", edgecolor="white", alpha=0.8)
    ax.axvline(df[col].mean(), color="red", linestyle="--", linewidth=1.5, label=f"Mean={df[col].mean():.1f}")
    ax.axvline(df[col].median(), color="orange", linestyle=":", linewidth=1.5)
    ax.set_title(col); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,5))
fig.suptitle("Disease Prevalence", fontsize=14, fontweight="bold")
for ax, col, labels in zip(axes, ["Disease","Diabetes","Heart_Disease"],
    [["No Disease","Disease"],["No Diabetes","Diabetes"],["No Heart Disease","Heart Disease"]]):
    counts = df[col].value_counts()
    ax.pie(counts, labels=labels, autopct="%1.1f%%",
           colors=["#4CAF50","#F44336"], startangle=90,
           wedgeprops={"edgecolor":"white","linewidth":2})
    ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14,11))
corr = df[num_cols + ["Disease","Diabetes","Heart_Disease"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlBu_r",
            center=0, ax=ax, linewidths=0.5, annot_kws={"size":7})
ax.set_title("Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

---
## Step 5: Statistical Analysis & Hypothesis Testing

In [ ]:
desc = df[num_cols].describe().T
desc["variance"] = df[num_cols].var()
desc["skewness"] = df[num_cols].skew()
print("Descriptive Statistics:")
print(desc[["mean","50%","std","variance","min","max","skewness"]].round(3))

In [ ]:
print("=== Independent t-tests ===
H0: No difference between groups
H1: Significant difference exists")
for col in ["Age","BMI","Blood_Glucose","Systolic_BP","Cholesterol","HbA1c"]:
    g0 = df[df["Disease"]==0][col]; g1 = df[df["Disease"]==1][col]
    t, p = stats.ttest_ind(g0, g1)
    sig = "REJECT H0" if p < 0.05 else "Fail to Reject"
    print(f"{col:18s} t={t:8.3f} p={p:.6f} -> {sig}")

print("
=== Chi-Square Tests ===")
for cat in ["Smoking_Status","Physical_Activity","Diet_Quality","Gender"]:
    ct = pd.crosstab(df[cat], df["Disease"])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    sig = "REJECT H0" if p < 0.05 else "Fail to Reject"
    print(f"{cat:22s} chi2={chi2:8.2f} p={p:.6f} -> {sig}")

---
## Step 6: Predictive Modelling

In [ ]:
le = LabelEncoder()
cat_cols = ["Gender","Smoking_Status","Alcohol_Use","Physical_Activity","Diet_Quality"]
for col in cat_cols:
    df[col+"_enc"] = le.fit_transform(df[col].astype(str))

drop_cols = ["PatientID","Diabetes","Heart_Disease"] + cat_cols + ["BMI_Category"] if "BMI_Category" in df.columns else ["PatientID","Diabetes","Heart_Disease"] + cat_cols
features = [c for c in df.columns if c not in drop_cols + ["Disease"] and c in df.columns]

X = df[features]; y = df["Disease"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train); X_test_sc = scaler.transform(X_test)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

### Model 1: Linear Regression

In [ ]:
lr_features = ["Age","BMI","Systolic_BP","Cholesterol","HbA1c","Creatinine","LDL","HDL","Triglycerides","Hemoglobin"]
Xlr = df[lr_features]; ylr = df["Blood_Glucose"]
Xlr_train, Xlr_test, ylr_train, ylr_test = train_test_split(Xlr, ylr, test_size=0.2, random_state=42)
sc_lr = StandardScaler()
Xlr_tr = sc_lr.fit_transform(Xlr_train); Xlr_te = sc_lr.transform(Xlr_test)
lrm = LinearRegression().fit(Xlr_tr, ylr_train)
ylr_pred = lrm.predict(Xlr_te)
print(f"R2={r2_score(ylr_test,ylr_pred):.4f}  RMSE={np.sqrt(mean_squared_error(ylr_test,ylr_pred)):.4f}")
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(ylr_test, ylr_pred, alpha=0.3, s=10, color="#2196F3")
ax.plot([ylr_test.min(),ylr_test.max()],[ylr_test.min(),ylr_test.max()],"r--")
ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
ax.set_title(f"Linear Regression: Blood Glucose (R2={r2_score(ylr_test,ylr_pred):.3f})")
plt.tight_layout(); plt.show()

### Model 2: Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_sc, y_train)
y_pred_lr = log_reg.predict(X_test_sc)
y_prob_lr = log_reg.predict_proba(X_test_sc)[:,1]
print(f"Accuracy: {accuracy_score(y_test,y_pred_lr):.4f}")
print(f"ROC-AUC : {roc_auc_score(y_test,y_prob_lr):.4f}")
print(f"CV Acc  : {cross_val_score(log_reg,X_train_sc,y_train,cv=5).mean():.4f}")
print()
print(classification_report(y_test, y_pred_lr, target_names=["No Disease","Disease"]))

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(18,5))
ConfusionMatrixDisplay(confusion_matrix(y_test,y_pred_lr),display_labels=["No Disease","Disease"]).plot(ax=axes[0],cmap="Blues",colorbar=False)
fpr,tpr,_ = roc_curve(y_test,y_prob_lr)
auc_lr = roc_auc_score(y_test,y_prob_lr)
axes[1].plot(fpr,tpr,color="#2196F3",linewidth=2,label=f"AUC={auc_lr:.3f}")
axes[1].plot([0,1],[0,1],"k--"); axes[1].legend(); axes[1].set_title("ROC Curve")
pd.Series(log_reg.coef_[0],index=features).abs().sort_values(ascending=False).head(12).plot(kind="bar",ax=axes[2],color="#FF9800",edgecolor="white")
axes[2].set_title("Feature Importances"); axes[2].tick_params(axis="x",rotation=45)
plt.tight_layout(); plt.show()

### Model 3: Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=6, min_samples_split=50, min_samples_leaf=20, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:,1]
print(f"Accuracy: {accuracy_score(y_test,y_pred_dt):.4f}")
print(f"ROC-AUC : {roc_auc_score(y_test,y_prob_dt):.4f}")
print()
print(classification_report(y_test, y_pred_dt, target_names=["No Disease","Disease"]))

In [ ]:
fig, ax = plt.subplots(figsize=(22,10))
plot_tree(dt, feature_names=features, class_names=["No Disease","Disease"],
          filled=True, max_depth=3, ax=ax, fontsize=8, impurity=False, proportion=True)
ax.set_title("Decision Tree Visualization", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

---
## Step 7: Model Comparison

In [ ]:
comp = pd.DataFrame({
    "Model":["Logistic Regression","Decision Tree"],
    "Accuracy":[accuracy_score(y_test,y_pred_lr), accuracy_score(y_test,y_pred_dt)],
    "ROC-AUC":[roc_auc_score(y_test,y_prob_lr), roc_auc_score(y_test,y_prob_dt)]
})
print(comp.to_string(index=False))

fpr_dt,tpr_dt,_ = roc_curve(y_test,y_prob_dt)
fig, ax = plt.subplots(figsize=(8,6))
ax.plot(fpr,tpr,color="#2196F3",linewidth=2,label=f"Logistic Reg (AUC={roc_auc_score(y_test,y_prob_lr):.3f})")
ax.plot(fpr_dt,tpr_dt,color="#F44336",linewidth=2,label=f"Decision Tree (AUC={roc_auc_score(y_test,y_prob_dt):.3f})")
ax.plot([0,1],[0,1],"k--"); ax.legend(); ax.set_title("Combined ROC Curves")
plt.tight_layout(); plt.show()

---
## Step 8: Conclusion

### Key Findings
1. **Age, BMI, Physical Activity, Smoking Status** are the top predictors of disease
2. **HbA1c** and **Blood Glucose** strongly linked to diabetes
3. **Logistic Regression** is the best model: Accuracy=67.5%, AUC=0.724
4. Hypothesis Testing confirms most clinical features are statistically significant
5. Gender shows NO significant association with disease (p=0.378)

### Societal Impact
- Early disease prediction can reduce healthcare costs significantly
- Enables preventive care and evidence-based clinical decision making

### Future Scope
- Add Random Forest, XGBoost for better accuracy
- Deploy as REST API (Flask/FastAPI)
- Integrate with Electronic Health Records (EHR)
- Add SHAP values for model explainability